In [1]:
!git clone https://github.com/Erick-FHP/broad-money-latam-analysis.git

Cloning into 'broad-money-latam-analysis'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 33 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 36.80 KiB | 4.09 MiB/s, done.
Resolving deltas: 100% (3/3), done.


# Cargando los datos crudos

In [9]:
import pandas as pd

PATH_CSV = "./broad-money-latam-analysis/data/raw/indicators.csv"

df = pd.read_csv(PATH_CSV)

# Eliminar columnas que no usarás
df.drop(columns=["Country Name", "Series Code"], inplace=True)

# Pasar a formato largo
df_long = (
    df.set_index(["Country Code", "Series Name"])
      .stack()
      .reset_index(name="value")
      .rename(columns={
          "Country Code": "country",
          "Series Name": "series",
          "level_2": "year"
      })
)

# Limpiar tipos
df_long["year"] = df_long["year"].str.extract(r"(\d{4})").astype(int)
df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")
df_long = df_long.sort_values(["country", "series", "year"]).reset_index(drop=True)
df_long = df_long.ffill()

print(df_long.head())
print(df_long.info())
print(df_long.describe(include='all'))

  country                  series  year      value
0     ARG  Broad money (% of GDP)  1976  19.054320
1     ARG  Broad money (% of GDP)  1977  22.472306
2     ARG  Broad money (% of GDP)  1978  24.540967
3     ARG  Broad money (% of GDP)  1979  25.867691
4     ARG  Broad money (% of GDP)  1980  24.955734
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1750 entries, 0 to 1749
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   country  1750 non-null   object 
 1   series   1750 non-null   object 
 2   year     1750 non-null   int64  
 3   value    1750 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 54.8+ KB
None
       country                  series         year         value
count     1750                    1750  1750.000000   1750.000000
unique       7                       5          NaN           NaN
top        ARG  Broad money (% of GDP)          NaN           NaN
freq       250              

# Creando nuevas columnas

In [13]:
df_long['decade'] = (df_long['year'] // 10) * 10
df_long['annual_change'] = (
    df_long
    .groupby(['country', 'series'])['value']
    .diff()
)
df_long['pct_change'] = (
    df_long
    .groupby(['country', 'series'])['value']
    .pct_change() * 100
)
df_long['rolling_5y'] = (
    df_long
    .groupby(['country', 'series'])['value']
    .transform(lambda x: x.rolling(5, min_periods=3).mean())
)
df_long['rolling_std_5y'] = (
    df_long
    .groupby(['country', 'series'])['value']
    .transform(lambda x: x.rolling(5, min_periods=3).std())
)


df_long.head()

,country,series,year,value,decade,change,annual_change,pct_change,rolling_5y,rolling_std_5y
0,ARG,Broad money (% of GDP),1976,19.054320,1970,NaN,NaN,NaN,NaN,NaN
1,ARG,Broad money (% of GDP),1977,22.472306,1970,3.417986,3.417986,17.938115,NaN,NaN
2,ARG,Broad money (% of GDP),1978,24.540967,1970,2.068661,2.068661,9.205381,22.022531,2.770839
3,ARG,Broad money (% of GDP),1979,25.867691,1970,1.326723,1.326723,5.406157,22.983821,2.968952
4,ARG,Broad money (% of GDP),1980,24.955734,1980,-0.911956,-0.911956,-3.525464,23.378204,2.718216


In [17]:
df_features = (
    df_long
    .pivot(
        index=['country', 'year'],
        columns='series',
        values='value'
    )
    .reset_index()
)

broad_money = "Broad money (% of GDP)"

world_bm = (
    df_features[df_features["country"] == "WLD"]
    [["year", broad_money]]
    .rename(columns={broad_money: "world_broad_money"})
)
df_features = df_features.merge(
    world_bm,
    on="year",
    how="left"
)
df_features["broad_money_gap_world"] = (
    df_features[broad_money]
    - df_features["world_broad_money"]
)
df_features["broad_money_gap_world_pct"] = (
    (
        df_features[broad_money]
        - df_features["world_broad_money"]
    )
    / df_features["world_broad_money"]
) * 100
df_features["broad_money_percentile"] = (
    df_features
    .groupby("country")[broad_money]
    .rank(pct=True)
)

df_features.head()

series,country,year,Broad money (% of GDP),Domestic credit to private sector (% of GDP),"GDP per capita, PPP (constant 2021 international $)",Gross capital formation (% of GDP),"Inflation, GDP deflator (annual %)",world_broad_money,broad_money_gap_world,broad_money_gap_world_pct,broad_money_percentile
0,ARG,1976,19.054320,13.562767,17.598293,30.729440,438.322780,61.832874,-42.778554,-69.184159,0.10
1,ARG,1977,22.472306,18.382910,17.598293,30.941689,159.427178,62.126561,-39.654255,-63.828183,0.30
2,ARG,1978,24.540967,20.557117,17.598293,27.800160,161.372172,65.392679,-40.851712,-62.471384,0.40
3,ARG,1979,25.867691,24.056228,17.598293,25.856769,147.377044,64.114462,-38.246771,-59.653891,0.52
4,ARG,1980,24.955734,25.395793,17.598293,25.257786,95.790425,62.948836,-37.993102,-60.355527,0.42


# Exportando el dataset final

In [18]:
df_long.to_csv(
    "./long.csv",
    index=False
)

df_features.to_csv(
    "./features.csv",
    index=False
)